# Module 3 — Classification and Regression

**Part A:** classification of `seller_rating_class` with Decision Trees, KNN and Naïve Bayes.
**Part B:** single and multiple regression on one continuous target.

Leak-free design:
- one stratified 80/20 train/test split (`random_state=42`), saved for Module 4;
- all tuning by stratified 5-fold CV on the **training set only**;
- imputation, log transform, scaling, encoding and discretization live **inside** the CV pipelines;
- the test set is used once, for the final evaluation.

In [1]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = str(ROOT / "src")
sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")  # parallel CV workers import m3_utils
import plot_style as ps
from m3_utils import PriorCorrectedKNN

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (OneHotEncoder, StandardScaler, FunctionTransformer, OrdinalEncoder,
                                   KBinsDiscretizer, label_binarize)
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.neighbors import KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix,
                             roc_curve, roc_auc_score, auc, mean_squared_error, r2_score)
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # KBinsDiscretizer: bins of width 0 on zero-inflated columns
warnings.filterwarnings("ignore", category=FutureWarning)

ps.setup()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
SEED = 42

df = pd.read_csv(ROOT / "data" / "processed" / "clean_base.csv", index_col="sp_id")
fs = json.load(open(ROOT / "data" / "processed" / "feature_sets.json"))
TARGET = fs["target"]
NUM, CAT = fs["numeric_model"], fs["categorical_model"]
LOGV = [c for c in NUM if c in fs["log1p_vars"]]
RAWV = [c for c in NUM if c not in LOGV]
CLASSES = ps.CLASS_ORDER  # ["low", "medium", "high"]
print(len(NUM), "numeric +", len(CAT), "categorical features")

19 numeric + 5 categorical features


# Part A — Classification of `seller_rating_class`
## A.1 Experimental design

**Attributes:** the Module 1 modelling set: 19 numeric + 5 categorical features. Redundant variables (|ρ_s| ≥ 0.9), `engine_displacement` (24.5% missing) and `new_used_profile` (deterministic function of `pct_is_new`) were removed in Module 1, without using the target. `sp_id` is an identifier and is never a feature.

**Evaluation emphasis:** the classes are imbalanced (55 / 27 / 19%), and a majority-class predictor reaches 0.55 accuracy while never predicting `low` or `high`. Hyperparameters are therefore selected by **macro-F1** (each class weighted equally); accuracy is reported alongside.

**Imbalance handling, treated as a hyperparameter for every model:**
- DT: `class_weight` ∈ {none, balanced}.
- NB: empirical vs. uniform class priors.
- KNN: plain vs. prior-corrected decision rule, which predicts argmax P(c|x)/P(c); see `src/m3_utils.py`.

In [2]:
X = df[NUM + CAT]
y = df[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
split = pd.Series("train", index=df.index, name="set")
split.loc[X_te.index] = "test"
split.to_csv(ROOT / "data" / "processed" / "split.csv")
print("train", X_tr.shape, "test", X_te.shape)
print(pd.concat({"train %": y_tr.value_counts(normalize=True), "test %": y_te.value_counts(normalize=True)}, axis=1).loc[CLASSES].round(3))

CV = StratifiedKFold(5, shuffle=True, random_state=SEED)
SCORING = {"f1_macro": "f1_macro", "accuracy": "accuracy"}

def cat_onehot():
    return Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

def results_table(gs, params):
    r = pd.DataFrame(gs.cv_results_)
    cols = [f"param_{p}" for p in params] + ["mean_test_f1_macro", "std_test_f1_macro", "mean_train_f1_macro", "mean_test_accuracy"]
    r = r[cols].rename(columns=lambda c: c.replace("param_", ""))
    return r.sort_values("mean_test_f1_macro", ascending=False)

train (11074, 24) test (2769, 24)
                     train %  test %
seller_rating_class                 
low                    0.186   0.186
medium                 0.547   0.547
high                   0.266   0.266


### Baselines (5-fold CV on the training set)

In [3]:
from sklearn.model_selection import cross_validate
base_rows = []
for name, strat in [("majority class", "most_frequent"), ("stratified random", "stratified")]:
    cvres = cross_validate(DummyClassifier(strategy=strat, random_state=SEED), X_tr, y_tr, cv=CV, scoring=SCORING)
    base_rows.append((name, cvres["test_f1_macro"].mean(), cvres["test_accuracy"].mean()))
baselines = pd.DataFrame(base_rows, columns=["baseline", "CV macro-F1", "CV accuracy"]).round(3)
baselines

,baseline,CV macro-F1,CV accuracy
0,majority class,0.236,0.547
1,stratified random,0.334,0.407


## A.2 Decision Tree

- **Preprocessing:** median imputation for numeric features and most-frequent + one-hot for categoricals. There is no scaling or log transform, because trees are invariant to monotonic transformations.
- **Pre-pruning grid:** criterion {gini, entropy} × `max_depth` {2, 3, 4, 5, 6, 8, 10, 12, None} × `min_samples_split` {2, 20, 100} × `min_samples_leaf` {1, 5, 20, 50} × `class_weight` {None, balanced}, which gives **432 configurations**.

In [4]:
dt_pre = ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM), ("cat", cat_onehot(), CAT)])
dt_pipe = Pipeline([("pre", dt_pre), ("dt", DecisionTreeClassifier(random_state=SEED))])
dt_grid = {
    "dt__criterion": ["gini", "entropy"],
    "dt__max_depth": [2, 3, 4, 5, 6, 8, 10, 12, None],
    "dt__min_samples_split": [2, 20, 100],
    "dt__min_samples_leaf": [1, 5, 20, 50],
    "dt__class_weight": [None, "balanced"],
}
t0 = time.time()
dt_gs = GridSearchCV(dt_pipe, dt_grid, cv=CV, scoring=SCORING, refit="f1_macro", return_train_score=True, n_jobs=-1, error_score="raise").fit(X_tr, y_tr)
print(f"{len(dt_gs.cv_results_['params'])} configs in {time.time() - t0:.0f}s; best:", dt_gs.best_params_, "CV macro-F1 = %.3f" % dt_gs.best_score_)
dt_res = results_table(dt_gs, ["dt__criterion", "dt__max_depth", "dt__min_samples_split", "dt__min_samples_leaf", "dt__class_weight"])
dt_res.to_csv(ps.TAB_DIR / "m3_dt_grid.csv", index=False)
dt_res.head(10).round(3)

432 configs in 51s; best: {'dt__class_weight': 'balanced', 'dt__criterion': 'gini', 'dt__max_depth': 5, 'dt__min_samples_leaf': 50, 'dt__min_samples_split': 2} CV macro-F1 = 0.419


,dt__criterion,dt__max_depth,dt__min_samples_split,dt__min_samples_leaf,dt__class_weight,mean_test_f1_macro,std_test_f1_macro,mean_train_f1_macro,mean_test_accuracy
263,gini,5,100,50,balanced,0.419,0.014,0.443,0.431
262,gini,5,20,50,balanced,0.419,0.014,0.443,0.431
261,gini,5,2,50,balanced,0.419,0.014,0.443,0.431
260,gini,5,100,20,balanced,0.419,0.011,0.445,0.431
258,gini,5,2,20,balanced,0.419,0.012,0.445,0.431
259,gini,5,20,20,balanced,0.419,0.012,0.445,0.431
252,gini,5,2,1,balanced,0.417,0.011,0.444,0.428
255,gini,5,2,5,balanced,0.417,0.010,0.443,0.428
256,gini,5,20,5,balanced,0.417,0.010,0.443,0.428
253,gini,5,20,1,balanced,0.417,0.010,0.443,0.428


In [5]:
# Marginal view of the grid: best CV macro-F1 per value of each hyperparameter
marg = {}
for p in ["dt__criterion", "dt__max_depth", "dt__min_samples_leaf", "dt__min_samples_split", "dt__class_weight"]:
    marg[p.replace("dt__", "")] = dt_res.groupby(dt_res[p].astype(str))["mean_test_f1_macro"].max().round(3).to_dict()
marg

{'criterion': {'entropy': 0.406, 'gini': 0.419},
 'max_depth': {'10': 0.412,
  '12': 0.411,
  '2': 0.392,
  '3': 0.395,
  '4': 0.367,
  '5': 0.419,
  '6': 0.411,
  '8': 0.409,
  'None': 0.411},
 'min_samples_leaf': {'1': 0.417, '20': 0.419, '5': 0.417, '50': 0.419},
 'min_samples_split': {'100': 0.419, '2': 0.419, '20': 0.419},
 'class_weight': {'None': 0.407, 'balanced': 0.419}}

### Overfitting control: unpruned vs. pre-pruned vs. post-pruned (cost-complexity)

- **Unpruned:** the fully grown tree, with the best criterion and class weighting.
- **Post-pruning:** minimal cost-complexity pruning. The candidate α values come from the pruning path of the unpruned tree on the training data: 40 values, evenly spaced in rank. Each α is then evaluated by the same 5-fold CV.

In [6]:
bp = dt_gs.best_params_
unpruned = Pipeline([("pre", dt_pre), ("dt", DecisionTreeClassifier(criterion=bp["dt__criterion"], class_weight=bp["dt__class_weight"], random_state=SEED))])
Xt_tr = dt_pre.fit(X_tr, y_tr).transform(X_tr)
path = DecisionTreeClassifier(criterion=bp["dt__criterion"], class_weight=bp["dt__class_weight"], random_state=SEED).cost_complexity_pruning_path(Xt_tr, y_tr)
alphas = np.unique(path.ccp_alphas[np.linspace(0, len(path.ccp_alphas) - 2, 40).astype(int)])
ccp_gs = GridSearchCV(unpruned, {"dt__ccp_alpha": alphas}, cv=CV, scoring=SCORING, refit="f1_macro",
                      return_train_score=True, n_jobs=-1, error_score="raise").fit(X_tr, y_tr)
ccp_res = pd.DataFrame(ccp_gs.cv_results_)[["param_dt__ccp_alpha", "mean_train_f1_macro", "mean_test_f1_macro", "std_test_f1_macro"]]
ccp_res.columns = ["ccp_alpha", "train F1", "CV F1", "CV F1 std"]
ccp_res["leaves"] = [Pipeline([("pre", dt_pre), ("dt", DecisionTreeClassifier(criterion=bp["dt__criterion"], class_weight=bp["dt__class_weight"], ccp_alpha=a, random_state=SEED))]).fit(X_tr, y_tr)["dt"].get_n_leaves() for a in ccp_res.ccp_alpha]
ccp_res.to_csv(ps.TAB_DIR / "m3_dt_ccp.csv", index=False)

from sklearn.model_selection import cross_validate as cvd
def cv_summary(est):
    r = cvd(est, X_tr, y_tr, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
    leaves = est.fit(X_tr, y_tr)[-1].get_n_leaves()
    return {"train macro-F1": r["train_f1_macro"].mean(), "CV macro-F1": r["test_f1_macro"].mean(),
            "CV accuracy": r["test_accuracy"].mean(), "leaves": leaves}

from sklearn.base import clone
dt_cmp = pd.DataFrame({
    "unpruned": cv_summary(clone(unpruned)),
    "pre-pruned (grid best)": cv_summary(clone(dt_gs.best_estimator_)),
    f"post-pruned (ccp_alpha={ccp_gs.best_params_['dt__ccp_alpha']:.2e})": cv_summary(clone(ccp_gs.best_estimator_)),
}).T
dt_cmp.to_csv(ps.TAB_DIR / "m3_dt_pruning_comparison.csv")
dt_cmp.round(3)

,train macro-F1,CV macro-F1,CV accuracy,leaves
unpruned,1.000,0.387,0.451,3350.0
pre-pruned (grid best),0.443,0.419,0.431,31.0
post-pruned (ccp_alpha=4.54e-04),0.552,0.401,0.412,93.0


In [7]:
# Figure: overfitting curves. Left: depth sweep (other params at the grid optimum). Right: ccp_alpha sweep.
r = pd.DataFrame(dt_gs.cv_results_)
sel = np.ones(len(r), bool)
for p in ["dt__criterion", "dt__min_samples_split", "dt__min_samples_leaf", "dt__class_weight"]:
    sel &= r[f"param_{p}"].astype(str) == str(bp[p])
d = r[sel].copy()
d["depth"] = d["param_dt__max_depth"].astype(str)
order = ["2", "3", "4", "5", "6", "8", "10", "12", "None"]
d = d.set_index("depth").loc[order]

fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.2))
ax = axes[0]
xs = np.arange(len(order))
ax.plot(xs, d.mean_train_f1_macro, marker="o", markersize=4, color=ps.CATEGORICAL[1], label="train")
ax.plot(xs, d.mean_test_f1_macro, marker="o", markersize=4, color=ps.CATEGORICAL[0], label="5-fold CV")
ax.fill_between(xs, d.mean_test_f1_macro - d.std_test_f1_macro, d.mean_test_f1_macro + d.std_test_f1_macro, color=ps.CATEGORICAL[0], alpha=0.15, linewidth=0)
ax.set_xticks(xs); ax.set_xticklabels(order); ax.set_xlabel("max_depth (other parameters at grid optimum)")
ax.set_ylabel("macro-F1"); ax.set_title("Pre-pruning: depth"); ax.legend(frameon=False)
ax = axes[1]
cc = ccp_res[ccp_res.ccp_alpha >= 1e-5]  # smaller alphas leave the tree unpruned (train F1 = 1); omitted for readability
ax.plot(cc.ccp_alpha, cc["train F1"], marker="o", markersize=3, color=ps.CATEGORICAL[1], label="train")
ax.plot(cc.ccp_alpha, cc["CV F1"], marker="o", markersize=3, color=ps.CATEGORICAL[0], label="5-fold CV")
ax.axvline(ccp_gs.best_params_["dt__ccp_alpha"], color=ps.NEUTRAL, linestyle=":", linewidth=1)
ax.set_xscale("log"); ax.set_xlabel("ccp_alpha (log scale)"); ax.set_title("Post-pruning: cost-complexity")
ax.legend(frameon=False)
fig.tight_layout()
ps.save(fig, "m3_dt_overfitting")

In [8]:
# Final DT: the better CV macro-F1 of pre- and post-pruning
dt_final = clone(dt_gs.best_estimator_) if dt_gs.best_score_ >= ccp_gs.best_score_ else clone(ccp_gs.best_estimator_)
dt_final.fit(X_tr, y_tr)
DT_LABEL = "pre-pruned" if dt_gs.best_score_ >= ccp_gs.best_score_ else "post-pruned"
print("final DT:", DT_LABEL, "| depth", dt_final["dt"].get_depth(), "| leaves", dt_final["dt"].get_n_leaves())

final DT: pre-pruned | depth 5 | leaves 31


**Decision Tree results.**
- **Best configuration:** gini, `max_depth=5`, `min_samples_leaf=50`, `class_weight="balanced"`, with CV macro-F1 0.419 ± 0.014.
- **Which parameters matter:** class weighting (0.419 balanced vs 0.407 unweighted) and depth. The criterion matters less (gini 0.419 vs entropy 0.406), and `min_samples_split` changes nothing once `min_samples_leaf` ≥ 20.
- **Overfitting control:**
  - The **unpruned** tree (3,350 leaves) memorizes the training data (train F1 = 1.00) but reaches only 0.387 in CV.
  - **Pre-pruning** to depth 5 (31 leaves) gives the best CV score (0.419) with a small train–CV gap (0.443 vs 0.419).
  - **Post-pruning** by cost-complexity peaks at α ≈ 4.5·10⁻⁴ (93 leaves, CV 0.401) and is worse than pre-pruning. Its train curve drops steeply while the CV curve barely moves: most of the tree's structure is noise.
- **Depth curve:** the CV score is not monotone in depth (a dip at depth 4). Beyond depth 5 the differences are within about 1 CV standard deviation, so the depth-5 optimum should be read as "shallow trees are enough", not as a sharp peak.

The final tree is the **pre-pruned depth-5 tree**.

## A.3 K-Nearest Neighbours

- **Preprocessing:** median imputation, log1p on the Module 1 skewed variables, `StandardScaler` (distances need comparable scales), and one-hot categoricals.
- **Grid:** k ∈ {1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51, 75, 101, 151, 201, 301} × p ∈ {1 (Manhattan), 2 (Euclidean)} × weights ∈ {uniform, distance} × decision rule ∈ {plain, prior-corrected}, which gives **128 configurations**. The range extends to k = 301 so that the optimum is not at the edge of the grid.

In [9]:
knn_pre = ColumnTransformer([
    ("log", Pipeline([("imp", SimpleImputer(strategy="median")), ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("sc", StandardScaler())]), LOGV),
    ("raw", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), RAWV),
    ("cat", cat_onehot(), CAT)])
knn_pipe = Pipeline([("pre", knn_pre), ("knn", PriorCorrectedKNN())])
K_LIST = [1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51, 75, 101, 151, 201, 301]
knn_grid = {"knn__n_neighbors": K_LIST, "knn__p": [1, 2], "knn__weights": ["uniform", "distance"],
            "knn__prior_correction": [False, True]}
t0 = time.time()
knn_gs = GridSearchCV(knn_pipe, knn_grid, cv=CV, scoring=SCORING, refit="f1_macro", return_train_score=True, n_jobs=-1, error_score="raise").fit(X_tr, y_tr)
print(f"{len(knn_gs.cv_results_['params'])} configs in {time.time() - t0:.0f}s; best:", knn_gs.best_params_, "CV macro-F1 = %.3f" % knn_gs.best_score_)
knn_res = results_table(knn_gs, ["knn__n_neighbors", "knn__p", "knn__weights", "knn__prior_correction"])
knn_res.to_csv(ps.TAB_DIR / "m3_knn_grid.csv", index=False)
knn_res.head(10).round(3)

128 configs in 144s; best: {'knn__n_neighbors': 151, 'knn__p': 1, 'knn__prior_correction': True, 'knn__weights': 'distance'} CV macro-F1 = 0.434


,knn__n_neighbors,knn__p,knn__weights,knn__prior_correction,mean_test_f1_macro,std_test_f1_macro,mean_train_f1_macro,mean_test_accuracy
107,151,1,distance,True,0.434,0.008,1.000,0.449
103,101,2,distance,True,0.434,0.009,1.000,0.448
115,201,1,distance,True,0.433,0.009,1.000,0.449
111,151,2,distance,True,0.433,0.005,1.000,0.448
106,151,1,uniform,True,0.432,0.008,0.444,0.447
114,201,1,uniform,True,0.432,0.008,0.441,0.447
119,201,2,distance,True,0.432,0.010,1.000,0.448
102,101,2,uniform,True,0.431,0.008,0.450,0.446
110,151,2,uniform,True,0.431,0.005,0.446,0.446
118,201,2,uniform,True,0.431,0.010,0.443,0.446


In [10]:
# Figure: CV macro-F1 and accuracy vs k (p fixed at its best value)
r = pd.DataFrame(knn_gs.cv_results_)
best_p = knn_gs.best_params_["knn__p"]
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.2), sharex=True)
combos = [("uniform", False), ("distance", False), ("uniform", True), ("distance", True)]
for i, (w, pc) in enumerate(combos):
    s = r[(r.param_knn__p == best_p) & (r.param_knn__weights == w) & (r.param_knn__prior_correction == pc)].sort_values("param_knn__n_neighbors")
    lab = f"{w}, {'prior-corrected' if pc else 'plain'}"
    for ax, m in zip(axes, ["mean_test_f1_macro", "mean_test_accuracy"]):
        ax.plot(s.param_knn__n_neighbors, s[m], marker="o", markersize=3, color=ps.CATEGORICAL[i], label=lab,
                linestyle="-" if not pc else "--")
for ax, t in zip(axes, ["CV macro-F1", "CV accuracy"]):
    TICKS = [1, 3, 9, 21, 51, 101, 301]
    ax.set_xscale("log"); ax.set_xticks(TICKS); ax.set_xticklabels(TICKS, fontsize=6); ax.minorticks_off()
    ax.set_xlabel("k (log scale)"); ax.set_title(f"{t} (p={best_p})")
axes[1].axhline(baselines.loc[0, "CV accuracy"], color=ps.NEUTRAL, linestyle=":", linewidth=1)
axes[1].text(1, baselines.loc[0, "CV accuracy"] + 0.005, "majority baseline", fontsize=6, color=ps.INK_2)
axes[0].legend(frameon=False, fontsize=6)
fig.tight_layout()
ps.save(fig, "m3_knn_k")
knn_final = knn_gs.best_estimator_

**KNN results.** The decision rule changes the picture completely:
- **Plain majority vote:** as k grows, macro-F1 *falls* (0.38 → 0.24) while accuracy *rises* to the majority baseline (0.55). Large neighbourhoods are dominated by `medium`, so the model degenerates into "always predict medium".
- **Prior-corrected rule:** macro-F1 instead *increases* with k, to a plateau of ≈ 0.43 for k = 101–201, and drops slightly at 301. Large neighbourhoods give stable class-frequency estimates, and the correction removes the majority bias.
- **Other parameters:** p and the weighting matter much less than the decision rule (≤ 0.003 at the optimum).

Best configuration: k = 151, Manhattan (p = 1), distance weights, prior-corrected, with CV macro-F1 0.434 ± 0.008. With distance weights the training F1 is 1.0 by construction (each training point is its own nearest neighbour at distance 0), so the train–CV gap is not an overfitting signal here.

## A.4 Naïve Bayes

Two variants, each matched to a feature representation:
- **GaussianNB** on the 19 numeric features: median imputation + log1p, to bring the skewed variables closer to the Gaussian class-conditional assumption, then `StandardScaler`. Categoricals are excluded, because a Gaussian on one-hot 0/1 columns is meaningless. Grid: `var_smoothing` ∈ 10^{−12…0} (13 values) × priors {empirical, uniform}.
  - *Why scale?* GaussianNB itself is scale-invariant, but `var_smoothing` adds ε × (largest feature variance) to *every* variance. In a first run without scaling, raw mileage (variance ≈ 2.7·10⁹) dominated: ε = 10⁻⁹ already added ≈ 2.7 to every variance and flattened all log/proportion features (CV macro-F1 dropped from 0.40 to 0.31). Standardizing makes the smoothing comparable across features.
- **CategoricalNB** on *all* 24 features: numeric features are discretized into quantile bins (`KBinsDiscretizer`, fit inside each training fold) and categoricals are ordinal-encoded. The category lists are the known levels, not data statistics. Grid: bins ∈ {3, 4, 5, 6, 8, 10} × `alpha` (Laplace smoothing) ∈ {0.01, 0.1, 0.5, 1, 2, 5} × `fit_prior` {true, false}, which gives 72 configurations.

In [11]:
gnb_pre = ColumnTransformer([
    ("log", Pipeline([("imp", SimpleImputer(strategy="median")), ("log", FunctionTransformer(np.log1p)), ("sc", StandardScaler())]), LOGV),
    ("raw", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), RAWV)])
gnb_pipe = Pipeline([("pre", gnb_pre), ("nb", GaussianNB())])
prior_emp = None
prior_uni = [1 / 3] * 3
gnb_gs = GridSearchCV(gnb_pipe, {"nb__var_smoothing": np.logspace(-12, 0, 13), "nb__priors": [prior_emp, prior_uni]},
                      cv=CV, scoring=SCORING, refit="f1_macro", return_train_score=True, n_jobs=-1, error_score="raise").fit(X_tr, y_tr)
gnb_res = results_table(gnb_gs, ["nb__var_smoothing", "nb__priors"])
gnb_res["nb__priors"] = gnb_res["nb__priors"].map(lambda v: "empirical" if v is None else "uniform")
gnb_res.to_csv(ps.TAB_DIR / "m3_gnb_grid.csv", index=False)
print("GaussianNB best:", gnb_gs.best_params_, "CV macro-F1 = %.3f" % gnb_gs.best_score_)

CATS = [sorted(df[c].dropna().unique()) for c in CAT]
MAX_CAT = max(len(c) for c in CATS)
def cnb_pipe(n_bins):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("bin", KBinsDiscretizer(n_bins=n_bins, encode="ordinal", strategy="quantile"))]), NUM),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ord", OrdinalEncoder(categories=CATS))]), CAT)])
    return Pipeline([("pre", pre), ("nb", CategoricalNB(min_categories=max(n_bins, MAX_CAT)))])

cnb_rows, cnb_best, cnb_best_score = [], None, -1
for nb in [3, 4, 5, 6, 8, 10]:
    g = GridSearchCV(cnb_pipe(nb), {"nb__alpha": [0.01, 0.1, 0.5, 1, 2, 5], "nb__fit_prior": [True, False]},
                     cv=CV, scoring=SCORING, refit="f1_macro", return_train_score=True, n_jobs=-1, error_score="raise").fit(X_tr, y_tr)
    rr = results_table(g, ["nb__alpha", "nb__fit_prior"]); rr.insert(0, "bins", nb)
    cnb_rows.append(rr)
    if g.best_score_ > cnb_best_score:
        cnb_best, cnb_best_score, cnb_best_bins = g.best_estimator_, g.best_score_, nb
cnb_res = pd.concat(cnb_rows).sort_values("mean_test_f1_macro", ascending=False)
cnb_res.to_csv(ps.TAB_DIR / "m3_cnb_grid.csv", index=False)
print("CategoricalNB best: bins =", cnb_best_bins, cnb_best["nb"].get_params()["alpha"], "fit_prior =", cnb_best["nb"].fit_prior,
      "CV macro-F1 = %.3f" % cnb_best_score)
print(cnb_res.groupby("bins")["mean_test_f1_macro"].max().round(3).to_dict())
pd.concat([gnb_res.head(5).assign(model="GaussianNB"), cnb_res.head(5).assign(model="CategoricalNB")]).round(4)

GaussianNB best: {'nb__priors': [0.3333333333333333, 0.3333333333333333, 0.3333333333333333], 'nb__var_smoothing': 0.1} CV macro-F1 = 0.404


CategoricalNB best: bins = 3 0.01 fit_prior = True CV macro-F1 = 0.416
{3: 0.416, 4: 0.413, 5: 0.412, 6: 0.414, 8: 0.412, 10: 0.412}


,nb__var_smoothing,nb__priors,mean_test_f1_macro,std_test_f1_macro,mean_train_f1_macro,mean_test_accuracy,model,bins,nb__alpha,nb__fit_prior
24,0.10,uniform,0.4040,0.0186,0.4070,0.4333,GaussianNB,NaN,NaN,NaN
23,0.01,uniform,0.4027,0.0185,0.4059,0.4300,GaussianNB,NaN,NaN,NaN
13,0.00,uniform,0.4025,0.0181,0.4058,0.4296,GaussianNB,NaN,NaN,NaN
14,0.00,uniform,0.4025,0.0181,0.4058,0.4296,GaussianNB,NaN,NaN,NaN
20,0.00,uniform,0.4025,0.0181,0.4058,0.4296,GaussianNB,NaN,NaN,NaN
0,NaN,NaN,0.4159,0.0147,0.4181,0.4629,CategoricalNB,3.0,0.01,True
2,NaN,NaN,0.4159,0.0147,0.4181,0.4631,CategoricalNB,3.0,0.10,True
10,NaN,NaN,0.4155,0.0120,0.4180,0.4744,CategoricalNB,3.0,5.00,True
4,NaN,NaN,0.4153,0.0149,0.4185,0.4635,CategoricalNB,3.0,0.50,True
6,NaN,NaN,0.4152,0.0159,0.4181,0.4648,CategoricalNB,3.0,1.00,True


### How well do the Naïve Bayes assumptions hold?

In [12]:
# (1) Conditional independence: Spearman correlation between feature pairs *within* each class (training set)
pairs = [("mileage_median", "pct_is_new"), ("price_median", "mileage_median"), ("n_listings", "pct_is_new"),
         ("fuel_tank_volume_median", "length_median"), ("horsepower_median", "fuel_tank_volume_median")]
ci = pd.DataFrame({c: [X_tr.loc[y_tr == c, a].corr(X_tr.loc[y_tr == c, b], method="spearman") for a, b in pairs] for c in CLASSES},
                  index=[f"{a} ~ {b}" for a, b in pairs]).round(2)
# (2) Gaussianity after log1p: per-class skewness of the transformed features
Xg = pd.DataFrame(gnb_pre.fit(X_tr).transform(X_tr), columns=LOGV + RAWV, index=X_tr.index)
sk = pd.DataFrame({c: Xg[y_tr == c].skew() for c in CLASSES})
print("features with |skew| > 1 in at least one class (after log1p):", int((sk.abs() > 1).any(axis=1).sum()), "of", len(sk))
ci

features with |skew| > 1 in at least one class (after log1p): 11 of 19


,low,medium,high
mileage_median ~ pct_is_new,-0.87,-0.87,-0.81
price_median ~ mileage_median,-0.80,-0.80,-0.86
n_listings ~ pct_is_new,0.71,0.74,0.74
fuel_tank_volume_median ~ length_median,0.78,0.75,0.75
horsepower_median ~ fuel_tank_volume_median,0.83,0.80,0.80


Both assumptions are clearly violated:
- **Conditional independence:** strong dependencies persist *within* each rating class (e.g. mileage ~ pct_is_new, and the vehicle-size block), so NB counts the same evidence several times and its probabilities are over-confident.
- **Gaussian class-conditionals:** even after log1p, many features remain skewed, zero-inflated or bimodal inside each class.

CategoricalNB avoids the shape assumption by discretizing, which is why it does better than GaussianNB here. NB remains a useful, very fast baseline whose *ranking* of classes (AUC) can be reasonable even when its probabilities are poorly calibrated.

In [13]:
nb_final, NB_NAME = (cnb_best, "CategoricalNB") if cnb_best_score >= gnb_gs.best_score_ else (gnb_gs.best_estimator_, "GaussianNB")
print("NB variant carried to the test comparison as 'best NB':", NB_NAME)

NB variant carried to the test comparison as 'best NB': CategoricalNB


**Naïve Bayes results.**
- **GaussianNB** (standardized log features): `var_smoothing` has no effect up to 10⁻³ (plateau). The best is uniform priors with ε = 0.1, at CV macro-F1 0.404. Uniform priors beat empirical ones (0.404 vs 0.391), for the same reason the balanced tree wins.
- **CategoricalNB:** coarse bins work best (3 quantile bins, CV 0.416, with 4–10 bins all at 0.412–0.414). `alpha` is irrelevant (≤ 0.001 difference), because every bin holds hundreds of sellers. Empirical priors (`fit_prior=True`) are best here.
- Both variants are evaluated on the test set below; CategoricalNB is the stronger by CV.

## A.5 Test-set evaluation (held-out 20%, used once)

In [14]:
models = {
    "Decision Tree": dt_final,
    "KNN": knn_final,
    "GaussianNB": gnb_gs.best_estimator_,
    "CategoricalNB": cnb_best,
}
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)

def evaluate(name, m):
    pred = m.predict(X_te)
    proba = m.predict_proba(X_te)
    cols = [list(m.classes_).index(c) for c in CLASSES]
    P = proba[:, cols]
    Y = label_binarize(y_te, classes=CLASSES)
    pr, rc, f1, sup = precision_recall_fscore_support(y_te, pred, labels=CLASSES, zero_division=0)
    row = {"model": name, "accuracy": accuracy_score(y_te, pred),
           "macro-F1": f1_score(y_te, pred, average="macro"), "weighted-F1": f1_score(y_te, pred, average="weighted"),
           "AUC macro (OvR)": roc_auc_score(Y, P, average="macro"), "AUC micro": roc_auc_score(Y, P, average="micro")}
    for i, c in enumerate(CLASSES):
        row[f"P {c}"], row[f"R {c}"], row[f"F1 {c}"] = pr[i], rc[i], f1[i]
    return row, pred, P

test_rows, preds, probas = [], {}, {}
for n, m in models.items():
    row, preds[n], probas[n] = evaluate(n, m)
    test_rows.append(row)
row, _, _ = evaluate("Baseline: majority", dummy); test_rows.append(row)
test_tab = pd.DataFrame(test_rows).set_index("model")
test_tab.to_csv(ps.TAB_DIR / "m3_test_metrics.csv")
test_tab.round(3)

,accuracy,macro-F1,weighted-F1,AUC macro (OvR),AUC micro,P low,R low,F1 low,P medium,R medium,F1 medium,P high,R high,F1 high
model,,,,,,,,,,,,,,
Decision Tree,0.459,0.437,0.466,0.629,0.630,0.287,0.368,0.322,0.617,0.430,0.507,0.410,0.583,0.481
KNN,0.449,0.433,0.460,0.642,0.735,0.281,0.457,0.348,0.639,0.418,0.505,0.398,0.506,0.446
GaussianNB,0.440,0.407,0.452,0.593,0.609,0.248,0.374,0.299,0.608,0.480,0.537,0.372,0.402,0.386
CategoricalNB,0.459,0.414,0.461,0.594,0.655,0.315,0.238,0.272,0.602,0.493,0.542,0.352,0.544,0.428
Baseline: majority,0.547,0.236,0.387,0.500,0.661,0.000,0.000,0.000,0.547,1.000,0.708,0.000,0.000,0.000


In [15]:
# CV vs. test macro-F1 (generalization check)
cv_scores = {"Decision Tree": max(dt_gs.best_score_, ccp_gs.best_score_), "KNN": knn_gs.best_score_,
             "GaussianNB": gnb_gs.best_score_, "CategoricalNB": cnb_best_score}
gen = pd.DataFrame({"CV macro-F1": cv_scores, "test macro-F1": test_tab["macro-F1"].drop("Baseline: majority")}).round(3)
gen["chosen configuration"] = [DT_LABEL + " " + str({k.replace('dt__', ''): v for k, v in (dt_gs.best_params_ if DT_LABEL == 'pre-pruned' else ccp_gs.best_params_).items()}),
                               str({k.replace('knn__', ''): v for k, v in knn_gs.best_params_.items()}),
                               str({k.replace('nb__', ''): (v if k != 'nb__priors' else ('uniform' if v else 'empirical')) for k, v in gnb_gs.best_params_.items()}),
                               f"bins={cnb_best_bins}, alpha={cnb_best['nb'].alpha}, fit_prior={cnb_best['nb'].fit_prior}"]
gen.to_csv(ps.TAB_DIR / "m3_cv_vs_test.csv")
gen

,CV macro-F1,test macro-F1,chosen configuration
Decision Tree,0.419,0.437,"pre-pruned {'class_weight': 'balanced', 'crite..."
KNN,0.434,0.433,"{'n_neighbors': 151, 'p': 1, 'prior_correction..."
GaussianNB,0.404,0.407,"{'priors': 'uniform', 'var_smoothing': 0.1}"
CategoricalNB,0.416,0.414,"bins=3, alpha=0.01, fit_prior=True"


In [16]:
# Figure: confusion matrices (row-normalized = recall per true class; counts annotated)
fig, axes = plt.subplots(1, 4, figsize=(ps.FULL_W, 2.0))
from matplotlib.colors import LinearSegmentedColormap
seq = LinearSegmentedColormap.from_list("seq_blue", ["#ffffff", "#cde2fb", "#6da7ec", "#2a78d6", "#104281"])
for ax, (n, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_te, pred, labels=CLASSES)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cmn, cmap=seq, vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.0%}", ha="center", va="center", fontsize=5.8,
                    color="white" if cmn[i, j] > 0.55 else ps.INK)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES, fontsize=6.5)
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES if ax is axes[0] else [], fontsize=6.5)
    ax.set_title(f"{n}\nmacro-F1 {test_tab.loc[n, 'macro-F1']:.3f}", fontsize=7)
    ax.set_xlabel("predicted", fontsize=6.5); ax.grid(False)
    for s_ in ax.spines.values(): s_.set_visible(False)
axes[0].set_ylabel("true", fontsize=6.5)
fig.tight_layout()
ps.save(fig, "m3_confusion")

In [17]:
# Figure: one-vs-rest ROC curves per class + micro/macro averages
fig, axes = plt.subplots(1, 4, figsize=(ps.FULL_W, 2.2), sharey=True)
Y = label_binarize(y_te, classes=CLASSES)
for ax, (n, P) in zip(axes, probas.items()):
    fprs, tprs = [], []
    for i, c in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
        fprs.append(fpr); tprs.append(tpr)
        ax.plot(fpr, tpr, color=ps.CLASS_COLORS[c], linewidth=1.5, label=f"{c} ({auc(fpr, tpr):.2f})")
    fpr_mi, tpr_mi, _ = roc_curve(Y.ravel(), P.ravel())
    grid = np.linspace(0, 1, 201)
    tpr_ma = np.mean([np.interp(grid, f, t) for f, t in zip(fprs, tprs)], axis=0)
    ax.plot(fpr_mi, tpr_mi, color=ps.CATEGORICAL[1], linewidth=1.2, linestyle="--", label=f"micro ({auc(fpr_mi, tpr_mi):.2f})")
    ax.plot(grid, tpr_ma, color=ps.INK, linewidth=1.2, linestyle=":", label=f"macro ({auc(grid, tpr_ma):.2f})")
    ax.plot([0, 1], [0, 1], color=ps.GRID, linewidth=1)
    ax.set_title(n, fontsize=7.5); ax.set_xlabel("FPR", fontsize=6.5)
    ax.legend(frameon=False, fontsize=5.5, loc="lower right", handlelength=1.5)
    ax.set_aspect("equal")
axes[0].set_ylabel("TPR", fontsize=6.5)
fig.tight_layout()
ps.save(fig, "m3_roc")

**Test-set results.**
- **Generalization:** every model beats the majority baseline by a wide margin in macro-F1 (0.41–0.44 vs 0.24). The CV and test macro-F1 agree within 0.02, so the selection was not overfitted to the CV folds.
- **Accuracy:** all models stay *below* the 0.547 majority baseline. That is the expected cost of optimizing macro-F1: they predict the minority classes `low` and `high` much more often.
- **Decision Tree and KNN are tied** (test macro-F1 0.437 vs 0.433; CV 0.419 vs 0.434, with CV std ≈ 0.01) and both beat the two NB variants (0.41).
- **ROC/AUC:** macro AUC is only 0.59–0.64. The rating is separable to a limited degree, and `high` is always the easiest class (AUC 0.62–0.69), consistent with the "small independent seller" signal from Modules 1–2. KNN ranks best (macro AUC 0.642, micro 0.735) because its large-k neighbourhood frequencies are smooth, well-ordered probability estimates. The tree's probabilities take only 31 distinct values (one per leaf), hence its stepped curves.
- **Confusion matrices:** `medium` is the main source of error, being spread over `low` and `high` (DT: 25% / 32% of true medium sellers). DT has the best `high` recall (0.58), KNN the best `low` recall (0.46). CategoricalNB is weakest on `low` (recall 0.24). GaussianNB matches the tree on `low` (0.37) but has the lowest `high` recall (0.40).

## A.6 Interpreting the Decision Tree

In [18]:
dt_model = dt_final["dt"]
fnames = dt_final["pre"].get_feature_names_out()
def original(f):
    f = f.split("__", 1)[1]
    for c in CAT:
        if f.startswith(c + "_"):
            return c
    return f
imp = pd.Series(dt_model.feature_importances_, index=[original(f) for f in fnames]).groupby(level=0).sum().sort_values(ascending=False)
imp.to_csv(ps.TAB_DIR / "m3_dt_importance.csv")
print(imp.round(3).head(12))

fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 3.3), gridspec_kw={"width_ratios": [2.6, 1]})
short = lambda f: f.split("__", 1)[1].replace("_median", "").replace("dominant_", "")
plot_tree(dt_model, max_depth=2, feature_names=[short(f) for f in fnames], class_names=list(dt_model.classes_),
          filled=False, impurity=False, proportion=True, rounded=True, fontsize=5, ax=axes[0], precision=2)
axes[0].set_title(f"Final decision tree, top 3 levels (of depth {dt_model.get_depth()}, {dt_model.get_n_leaves()} leaves)\n"
                  "value = class proportions [high, low, medium]", fontsize=7)
top = imp.head(10)[::-1]
axes[1].barh([t.replace("_median", "").replace("dominant_", "") for t in top.index], top.values, color=ps.CATEGORICAL[0], height=0.6)
axes[1].set_xlabel("importance (impurity decrease)", fontsize=6.5); axes[1].set_title("Top-10 features", fontsize=7.5)
axes[1].tick_params(axis="y", labelsize=6.5); axes[1].grid(axis="y", visible=False)
fig.tight_layout()
ps.save(fig, "m3_dt_tree")

n_listings               0.617
price_median             0.126
pct_is_new               0.073
mileage_median           0.059
horsepower_median        0.027
height_median            0.021
savings_amount_median    0.016
franchise_status         0.016
daysonmarket_median      0.014
front_legroom_median     0.013
dominant_body_type       0.008
listing_span_days        0.006
dtype: float64


In [19]:
# Top decision rules: the leaves with the highest purity among sufficiently large leaves (training data)
tr = dt_model.tree_
Xt = dt_final["pre"].transform(X_tr)
leaf_id = dt_model.apply(Xt)
def rule_of(leaf):
    node_ind = dt_model.decision_path(Xt[leaf_id == leaf][:1]).indices
    conds = []
    for nd in node_ind[:-1]:
        f, th = tr.feature[nd], tr.threshold[nd]
        val = Xt[leaf_id == leaf][0, f]
        conds.append(f"{short(fnames[f])} {'<=' if val <= th else '>'} {th:.2f}")
    return " AND ".join(conds)
leaves = pd.DataFrame({"leaf": leaf_id, "y": y_tr.values}).groupby("leaf")["y"].agg(
    n="size", majority=lambda s: s.value_counts().index[0], purity=lambda s: s.value_counts(normalize=True).iloc[0])
leaves = leaves[leaves.n >= 50].sort_values("purity", ascending=False)
top_rules = []
for cls in CLASSES:
    for lf, rw in leaves[leaves.majority == cls].head(2).iterrows():
        top_rules.append({"class": cls, "n (train)": rw.n, "purity": round(rw.purity, 3), "rule": rule_of(lf)})
top_rules = pd.DataFrame(top_rules)
top_rules.to_csv(ps.TAB_DIR / "m3_dt_top_rules.csv", index=False)
pd.set_option("display.max_colwidth", 200)
top_rules

,class,n (train),purity,rule
0,low,290,0.500,n_listings > 103.50 AND price <= 26596.75 AND horsepower <= 191.25 AND front_legroom > 42.12 AND pct_is_new <= 0.70
1,medium,58,0.948,n_listings > 103.50 AND price > 26596.75 AND n_listings > 228.50 AND price > 36503.00 AND height <= 63.20
2,medium,280,0.786,n_listings > 103.50 AND price > 26596.75 AND n_listings <= 228.50 AND pct_is_new <= 0.62 AND mileage > 13814.50
3,high,279,0.652,n_listings <= 103.50 AND n_listings <= 17.50 AND mileage <= 79007.25 AND franchise_status_independent > 0.50 AND body_type_sedan <= 0.50
4,high,53,0.604,n_listings <= 103.50 AND n_listings <= 17.50 AND mileage > 79007.25 AND savings_amount > 857.50 AND price > 15849.75


**Reading the tree.** The tree uses `class_weight="balanced"`, so the class proportions shown in the plot are *weighted* (the root shows 1/3 each).
- **Root split: seller size.** The root splits on `n_listings ≤ 103.5`, and `n_listings` alone accounts for 62% of the total impurity decrease. Price (13%), `pct_is_new` (7%) and mileage (6%) follow; all other features are marginal.
- **Small sellers lean `high`.** This is the clearest pattern, and it repeats Modules 1–2. The purest `high` leaf (279 training sellers, 65% `high`, against a 27% base rate) is:
  - very small sellers (≤ 17 listings),
  - with a moderate-mileage stock (≤ 79k miles),
  - independent, and not mainly selling sedans.
- **Large, expensive sellers are `medium`:** more than 228 listings and a median price above $36.5k gives 95% `medium` (58 sellers).
- **`low` never gets a pure leaf.** Its best leaf (large sellers with cheaper, lower-powered stock) is only 50% `low`. The data contains no clear profile of a low-rated seller, which explains the weak `low` precision of all models.

The next section covers two points: the share of non-adjacent errors (low ↔ high) and the errors by Module 2 segment.

## A.7 Error analysis

In [20]:
# Where do the errors concentrate? (1) Confusions between adjacent classes; (2) performance by Module 2 seller segment
best_name = test_tab.drop("Baseline: majority")["macro-F1"].idxmax()
cm = pd.DataFrame(confusion_matrix(y_te, preds[best_name], labels=CLASSES), index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES])
print("best test macro-F1:", best_name); print(cm)
err = pd.Series(preds[best_name] != y_te.values, index=y_te.index)
print("\nshare of errors that are low<->high (non-adjacent):",
      round(((y_te == "low") & (preds[best_name] == "high")).sum() / err.sum() + ((y_te == "high") & (preds[best_name] == "low")).sum() / err.sum(), 3))

clusters = pd.read_csv(ROOT / "data" / "processed" / "cluster_labels.csv", index_col="sp_id")["kmeans4"].loc[y_te.index]
seg = pd.DataFrame({
    "test sellers": clusters.value_counts().sort_index().values,
    "accuracy": [accuracy_score(y_te[clusters == k], preds[best_name][(clusters == k).values]) for k in sorted(clusters.unique())],
    "macro-F1": [f1_score(y_te[clusters == k], preds[best_name][(clusters == k).values], average="macro") for k in sorted(clusters.unique())],
    "high share (true)": [(y_te[clusters == k] == "high").mean() for k in sorted(clusters.unique())],
}, index=[f"C{k}" for k in sorted(clusters.unique())])
seg.to_csv(ps.TAB_DIR / "m3_error_by_segment.csv")
seg.round(3)

best test macro-F1: Decision Tree
             pred low  pred medium  pred high
true low          190          189        137
true medium       381          652        483
true high          92          215        430

share of errors that are low<->high (non-adjacent): 0.153


,test sellers,accuracy,macro-F1,high share (true)
C0,1015,0.419,0.412,0.201
C1,701,0.504,0.451,0.247
C2,267,0.464,0.426,0.397
C3,786,0.471,0.388,0.323


**Error analysis.**
- **Mostly adjacent errors:** 85% of the best model's errors are between adjacent classes (low ↔ medium or medium ↔ high). Only 15% confuse `low` with `high`. With the same marginals, independent predictions would give 21% (notebook 05, §5), so the models capture the *ordinal direction* of the rating only in part.
- **Probable cause:** `seller_rating_class` is a discretization of a continuous `seller_rating`, so sellers near the class thresholds are intrinsically ambiguous. Combined with the weak feature–target relation (Module 1: ε² ≤ 0.08) and the absence of rating-aligned clusters (Module 2: ARI ≤ 0.025), this sets a low ceiling for any classifier using these features.
- **By Module 2 segment:** performance is fairly uniform (macro-F1 0.39–0.45). The model does best on the mid-market used dealers (C1) and worst on the small budget independents (C3), where the classes overlap most. *Caveat:* per-segment macro-F1 also depends on each segment's class mix, so it is descriptive only.

# Part B — Regression
## B.1 Task setup

**Target: `price_median`** (the seller's median listing price), modelled on the **log scale** (`log1p`).
- Price is the central economic variable of a seller's inventory and is strongly related to the other attributes (Module 1 heatmap).
- The raw distribution is right-skewed (skew 3.4); the log makes it nearly symmetric (−0.4) and turns multiplicative effects (e.g. "−x% per 10k miles") into additive ones.
- No predictor is a derivation of price: `savings_amount` is a discount, not a price statistic. Errors on the log scale are relative errors (RMSE 0.2 ≈ ±20%).

**Same split as Part A** (the 80/20 train/test indices), so all regression models are compared on identical data. Tuning uses 5-fold CV (`KFold`, shuffled) on the training set, scored by MSE. Missing predictor values are median-imputed inside the pipelines.

In [21]:
yr = np.log1p(df.price_median)
yr_tr, yr_te = yr.loc[X_tr.index], yr.loc[X_te.index]
num_cands = [c for c in df.select_dtypes("number").columns if c != "price_median"]
corr_price = df.loc[X_tr.index, num_cands].corrwith(yr_tr, method="spearman").sort_values(key=abs, ascending=False)
corr_price.round(3).head(10)

mileage_median        -0.822
year_median            0.817
vehicle_age_median    -0.813
pct_is_new             0.719
owner_count_median    -0.715
n_listing_days         0.607
n_listings             0.606
width_median           0.568
horsepower_median      0.494
back_legroom_median    0.487
dtype: float64

## B.2 Single linear regression: log price ~ mileage

`mileage_median` has the strongest correlation with price on the training set (ρ_s ≈ −0.82; year and vehicle age are close behind, but were excluded in Module 1 as redundant with mileage). It is also the most direct measure of vehicle wear, which is the textbook driver of used-car prices.

In [22]:
KF = KFold(5, shuffle=True, random_state=SEED)
def reg_eval(name, model, cols):
    model.fit(df.loc[X_tr.index, cols], yr_tr)
    p_tr, p_te = model.predict(df.loc[X_tr.index, cols]), model.predict(df.loc[X_te.index, cols])
    return {"model": name, "predictors": ", ".join(c.replace("_median", "") for c in cols),
            "train MSE": mean_squared_error(yr_tr, p_tr), "test MSE": mean_squared_error(yr_te, p_te),
            "train R2": r2_score(yr_tr, p_tr), "test R2": r2_score(yr_te, p_te),
            "test R2 ($ scale)": r2_score(np.expm1(yr_te), np.expm1(p_te))}, p_te

slr = Pipeline([("imp", SimpleImputer(strategy="median")), ("lr", LinearRegression())])
slr_row, slr_pred = reg_eval("Single linear (OLS)", slr, ["mileage_median"])
b0, b1 = slr["lr"].intercept_, slr["lr"].coef_[0]
print(f"log1p(price) = {b0:.3f} {b1 * 1e4:+.4f} * (mileage / 10,000)  ->  {(np.exp(b1 * 1e4) - 1) * 100:.1f}% price per +10k miles")
pd.Series(slr_row).to_frame().T

log1p(price) = 10.404 -0.1035 * (mileage / 10,000)  ->  -9.8% price per +10k miles


,model,predictors,train MSE,test MSE,train R2,test R2,test R2 ($ scale)
0,Single linear (OLS),mileage,0.100069,0.097804,0.740359,0.746615,0.481335


In [23]:
# Figure: fitted line + residuals
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.3))
ax = axes[0]
hb = ax.hexbin(df.loc[X_tr.index, "mileage_median"] / 1000, yr_tr, gridsize=45, cmap=seq, mincnt=1, linewidths=0.1, bins="log")
xs = np.linspace(0, df.mileage_median.max(), 100)
ax.plot(xs / 1000, b0 + b1 * xs, color=ps.CATEGORICAL[1], linewidth=2, label=f"OLS fit (test R² = {slr_row['test R2']:.2f})")
ax.set_xlabel("median mileage (k mi)"); ax.set_ylabel("log1p(median price)"); ax.legend(frameon=False, loc="upper right")
ax.set_title("Single regression (training data)")
cb = fig.colorbar(hb, ax=ax, pad=0.01); cb.set_label("sellers (log scale)", fontsize=6.5); cb.outline.set_visible(False)
ax = axes[1]
res = yr_te - slr_pred
ax.hexbin(slr_pred, res, gridsize=45, cmap=seq, mincnt=1, linewidths=0.1, bins="log")
ax.axhline(0, color=ps.INK_2, linewidth=0.8)
ax.set_xlabel("fitted log1p(price)"); ax.set_ylabel("residual"); ax.set_title("Residuals vs fitted (test)")
fig.tight_layout()
ps.save(fig, "m3_slr")
print("residual mean by mileage group (test):")
print(res.groupby(pd.cut(df.loc[X_te.index, "mileage_median"], [-1, 100, 20000, 60000, 120000, 1e6])).agg(["mean", "std", "size"]).round(3))

residual mean by mileage group (test):
                        mean    std  size
mileage_median                           
(-1.0, 100.0]         -0.053  0.245   750
(100.0, 20000.0]       0.089  0.361   296
(20000.0, 60000.0]     0.032  0.277   659
(60000.0, 120000.0]   -0.028  0.303   667
(120000.0, 1000000.0]  0.026  0.420   397


**Single regression results.**
- **Fit:** every +10,000 miles of median mileage lowers the median price by ≈ 9.8%. The fit explains 75% of the variance of log price, identically on train (0.740) and test (0.747), so there is no overfitting.
- **Price scale:** R² is only 0.48 in dollars, because errors in log space become large dollar errors for expensive inventories.
- **Residuals show where the straight line fails:**
  - The new-car sellers pile up at zero mileage, with a wide spread of prices that mileage cannot explain (a vertical stripe at the right of the residual plot).
  - Nearly-new used stock (100–20k miles) is under-predicted (mean residual +0.09).
  - The residual spread grows with mileage (std 0.25 → 0.42): old inventories differ in ways mileage does not capture (vehicle class, power).

These patterns motivate adding predictors and non-linear models.

## B.3 Multiple regression: same target, 4 predictors

Predictors: `mileage_median` (wear), `horsepower_median` (power/performance), `width_median` (vehicle size/class) and `pct_is_new` (share of new cars in the inventory). They describe different aspects of a seller's stock, and none is a derivation of price. Multicollinearity is mild: VIFs on the training set are below 2.5 (next cell), even though mileage and `pct_is_new` are correlated (ρ_s = −0.86). Year and vehicle age were excluded in Module 1 as redundant with mileage.

Models and tuning grids:
1. **Multiple linear regression (OLS):** no hyperparameters.
2. **Decision Tree Regressor:** `max_depth` {2, 3, 4, 5, 6, 8, 10, 12, 15, None} × `min_samples_leaf` {1, 5, 10, 20, 50, 100}, which gives 60 configurations.
3. **KNN Regressor** (standardized inputs): k {1, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100} × p {1, 2} × weights {uniform, distance}, which gives 44 configurations.

In [24]:
RP = ["mileage_median", "horsepower_median", "width_median", "pct_is_new"]
Xv = df.loc[X_tr.index, RP].fillna(df.loc[X_tr.index, RP].median())
vif = {c: 1 / (1 - LinearRegression().fit(Xv.drop(columns=c), Xv[c]).score(Xv.drop(columns=c), Xv[c])) for c in RP}
print("VIF (train):", {k: round(v, 2) for k, v in vif.items()})

dtr_gs = GridSearchCV(Pipeline([("imp", SimpleImputer(strategy="median")), ("dt", DecisionTreeRegressor(random_state=SEED))]),
                      {"dt__max_depth": [2, 3, 4, 5, 6, 8, 10, 12, 15, None], "dt__min_samples_leaf": [1, 5, 10, 20, 50, 100]},
                      cv=KF, scoring="neg_mean_squared_error", return_train_score=True, n_jobs=-1, error_score="raise").fit(df.loc[X_tr.index, RP], yr_tr)
knr_gs = GridSearchCV(Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("knn", KNeighborsRegressor())]),
                      {"knn__n_neighbors": [1, 3, 5, 7, 10, 15, 20, 30, 50, 75, 100], "knn__p": [1, 2], "knn__weights": ["uniform", "distance"]},
                      cv=KF, scoring="neg_mean_squared_error", return_train_score=True, n_jobs=-1, error_score="raise").fit(df.loc[X_tr.index, RP], yr_tr)
for name, g in [("DT regressor", dtr_gs), ("KNN regressor", knr_gs)]:
    rr = pd.DataFrame(g.cv_results_)
    keep = [c for c in rr.columns if c.startswith("param_")] + ["mean_test_score", "mean_train_score"]
    rr = rr[keep].assign(cv_MSE=lambda d: -d.mean_test_score, train_MSE=lambda d: -d.mean_train_score).drop(columns=["mean_test_score", "mean_train_score"])
    rr.sort_values("cv_MSE").to_csv(ps.TAB_DIR / f"m3_{name.split()[0].lower()}_reg_grid.csv", index=False)
    print(name, "best:", g.best_params_, "CV MSE = %.4f" % -g.best_score_)

rows, reg_preds = [slr_row], {"Single linear (mileage)": slr_pred}
for name, model in [("Multiple linear (OLS)", Pipeline([("imp", SimpleImputer(strategy="median")), ("lr", LinearRegression())])),
                    ("Decision Tree regressor", clone(dtr_gs.best_estimator_)),
                    ("KNN regressor", clone(knr_gs.best_estimator_))]:
    rw, p = reg_eval(name, model, RP); rows.append(rw); reg_preds[name] = p
reg_tab = pd.DataFrame(rows).set_index("model")
reg_tab["gap (test-train MSE)"] = reg_tab["test MSE"] - reg_tab["train MSE"]
reg_tab.to_csv(ps.TAB_DIR / "m3_regression_results.csv")
reg_tab.round(4)

VIF (train): {'mileage_median': 2.41, 'horsepower_median': 1.92, 'width_median': 2.05, 'pct_is_new': 2.42}


DT regressor best: {'dt__max_depth': 12, 'dt__min_samples_leaf': 20} CV MSE = 0.0374
KNN regressor best: {'knn__n_neighbors': 30, 'knn__p': 1, 'knn__weights': 'distance'} CV MSE = 0.0337


,predictors,train MSE,test MSE,train R2,test R2,test R2 ($ scale),gap (test-train MSE)
model,,,,,,,
Single linear (OLS),mileage,0.1001,0.0978,0.7404,0.7466,0.4813,-0.0023
Multiple linear (OLS),"mileage, horsepower, width, pct_is_new",0.0462,0.0479,0.8802,0.8758,0.7634,0.0018
Decision Tree regressor,"mileage, horsepower, width, pct_is_new",0.0284,0.0376,0.9263,0.9026,0.8691,0.0092
KNN regressor,"mileage, horsepower, width, pct_is_new",0.0000,0.0337,1.0000,0.9128,0.8735,0.0336


In [25]:
mlr = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("lr", LinearRegression())]).fit(df.loc[X_tr.index, RP], yr_tr)
print("standardized OLS coefficients (effect of +1 SD on log price):",
      dict(zip(RP, np.round(mlr["lr"].coef_, 3))))

standardized OLS coefficients (effect of +1 SD on log price): {'mileage_median': -0.482, 'horsepower_median': 0.208, 'width_median': 0.039, 'pct_is_new': 0.016}


In [26]:
# Figure: predicted vs actual on the test set
fig, axes = plt.subplots(1, 4, figsize=(ps.FULL_W, 2.0), sharex=True, sharey=True)
lo, hi = yr_te.min(), yr_te.max()
for ax, (n, p) in zip(axes, reg_preds.items()):
    ax.hexbin(yr_te, p, gridsize=35, cmap=seq, mincnt=1, linewidths=0.1, bins="log", extent=(lo, hi, lo, hi))
    ax.plot([lo, hi], [lo, hi], color=ps.CATEGORICAL[1], linewidth=1)
    ax.set_title(f"{n}\nR² {r2_score(yr_te, p):.3f}, MSE {mean_squared_error(yr_te, p):.3f}", fontsize=6.5)
    ax.set_xlabel("actual log price", fontsize=6.5)
axes[0].set_ylabel("predicted log price", fontsize=6.5)
fig.tight_layout()
ps.save(fig, "m3_regression_pred")

**Multiple regression results.**
- **Multiple linear (OLS):** adding horsepower, width and `pct_is_new` halves the error of the single regression (test MSE 0.098 → 0.048; R² 0.747 → 0.876), with a negligible generalization gap. The standardized coefficients show that mileage (−0.48 per SD) and horsepower (+0.21) carry almost all of the effect. Width and `pct_is_new` add little once mileage is known, because they partly duplicate it.
- **Decision Tree regressor:** best at depth 12 with ≥ 20 sellers per leaf (CV-selected from 60 configurations). Test R² is 0.903 (MSE 0.038); the train–test gap is small (0.009), thanks to the leaf-size constraint.
- **KNN regressor:** best at k = 30, Manhattan distance, distance weights (CV-selected from 44 configurations). It has the **lowest test error** (R² 0.913, MSE 0.034). Its train MSE of 0 comes from distance weighting (each training point predicts itself), not from a failure to generalize: the test error is still the best.
- **Why non-linearity helps:** both non-linear models capture the effects the residuals revealed, such as the zero-mileage new-car regime and the power × size interactions. That gains another 0.03–0.04 of R² over OLS.

# Part C — Discussion: which algorithm is best?

**Classification.** No model is clearly best on accuracy metrics: DT and KNN are statistically tied (≈ 0.43–0.44 test macro-F1), and both NB variants trail (≈ 0.41).

| Perspective | Decision Tree | KNN | Naïve Bayes |
|---|---|---|---|
| Test macro-F1 / macro AUC | 0.437 / 0.63 | 0.433 / **0.64** | 0.41 / 0.59 |
| Class strength | best `high` recall | best `low` recall | CNB weak on `low`, GNB weak on `high` |
| Generalization gap | small after pruning | n/a (distance weights) | very small |
| Interpretability | **explicit rules** | none | per-feature likelihoods |
| Robustness to skew / outliers | invariant to monotone transforms | needs log + scaling | needs log (Gaussian) or bins |
| Cost | **fast fit, trivial prediction** | no fit, but 151-NN search per prediction | fastest |
| Assumptions | none on distributions | meaningful distance | independence and shape (violated) |

→ **The pruned Decision Tree is the best overall choice.** It matches KNN's macro-F1 and adds readable rules: small, independent, used-car sellers are rated higher. It is also the cheapest model to deploy. KNN is preferable only if well-ranked probabilities (AUC) are the priority.

**Regression.**
- **Lowest error:** KNN (R² 0.913).
- **Best trade-off:** the Decision Tree regressor (R² 0.903, small gap, interpretable splits, fast prediction).
- **Interpretation:** the multiple OLS model is the most interpretable (each coefficient is a % price effect) and still explains 88% of the variance.

**Overall:**
- Price is strongly determined by inventory characteristics, especially wear and power.
- The seller *rating* is only weakly determined by them. This is a property of the data (Modules 1–2), not a failure of the algorithms.

In [27]:
# Summary of the numbers quoted in Part C
summary = {
    "classification (test)": test_tab[["accuracy", "macro-F1", "AUC macro (OvR)", "R low", "R medium", "R high"]].round(3),
    "regression (test)": reg_tab[["test MSE", "test R2", "gap (test-train MSE)"]].round(4),
}
for k, v in summary.items():
    print(f"\n== {k}\n{v}")


== classification (test)
                    accuracy  macro-F1  AUC macro (OvR)  R low  R medium  R high
model                                                                           
Decision Tree          0.459     0.437            0.629  0.368     0.430   0.583
KNN                    0.449     0.433            0.642  0.457     0.418   0.506
GaussianNB             0.440     0.407            0.593  0.374     0.480   0.402
CategoricalNB          0.459     0.414            0.594  0.238     0.493   0.544
Baseline: majority     0.547     0.236            0.500  0.000     1.000   0.000

== regression (test)
                         test MSE  test R2  gap (test-train MSE)
model                                                           
Single linear (OLS)        0.0978   0.7466               -0.0023
Multiple linear (OLS)      0.0479   0.8758                0.0018
Decision Tree regressor    0.0376   0.9026                0.0092
KNN regressor              0.0337   0.9128                0.